In [ ]:
import os


In [ ]:
import json
import numpy as np
import pandas as pd
from copy import deepcopy
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ultralytics import YOLO


In [ ]:
# Training parameters
CLASSES = [
    'Scratches on the circuit board',
    'Scratches on components',
    'Missing/lost component',
    'Damaged component pins',
    'Faulty components'
]
ROOT_DIR = os.path.join(os.getcwd(), 'datasets/PCBdatasets')
IMG_SIZE = 512
EPOCHS = 100
BATCH_SIZE = 8

# Evaluation parameters
CONF_THRESH = 0.5
NMS_THRESH = 0.25
IOU_THRESH = 0.5


# I. Model Training

In [ ]:
# Train YOLOv8
YOLO(model='yolov8m.pt', task='detect').train(
    data=os.path.join(ROOT_DIR, 'data.yaml'), epochs=EPOCHS, batch=BATCH_SIZE, imgsz=IMG_SIZE,
    mosaic=0.8, device='1', project=os.path.join(os.getcwd(), 'runs/yolov8m')
)


In [ ]:
# Train YOLOv11
YOLO(model='yolo11m.pt', task='detect').train(
    data=os.path.join(ROOT_DIR, 'data.yaml'), epochs=EPOCHS, batch=BATCH_SIZE, imgsz=IMG_SIZE,
    mosaic=0.8, device='1', project=os.path.join(os.getcwd(), 'runs/yolo11m')
)


# II. Evaluation

In [ ]:
def load_yolo_gts_and_preds(gt_path, pred_path, class_names):
    gts = {cls_name: {} for cls_name in class_names} # cls: {img_name: bbox}
    for label_path in Path(gt_path).glob('*.txt'):
        for cls_name in class_names:
            gts[cls_name][label_path.stem] = []
        with open(label_path, mode='r') as file:
            content = file.read()
            for label in [l.strip().split(' ') for l in content.strip().split('\n')]:
                gts[class_names[int(label[0])]][label_path.stem].append(format_bbox(np.array(label[1:], dtype=np.float_)))
        for cls_name in class_names:
            bbox = np.array(gts[cls_name][label_path.stem], dtype=np.float32)
            gts[cls_name][label_path.stem] = {
                'bbox': bbox,
                'check': np.zeros(len(bbox), dtype=np.bool_)
            }

    preds = {cls_name: [] for cls_name in class_names} # cls: {img_name: bbox}
    for label_path in Path(pred_path).glob('*.txt'):
        with open(label_path, mode='r') as file:
            content = file.read()
            label = np.array([l.split(' ') for l in content.strip().split('\n')], dtype=np.float32)
            with open(label_path, mode='r') as file:
                content = file.read()
                for label in [l.split(' ') for l in content.strip().split('\n')]:
                    preds[class_names[int(label[0])]].append({
                        'stem': label_path.stem,
                        'conf': float(label[-1]),
                        'bbox': np.array(format_bbox(np.array(label[1:-1], dtype=np.float_)))
                    })

    return gts, preds


def yolo_evaluation(gt_path, pred_path, class_names, iou_thresh, project):
    exp_path = increment_path(Path(project) / 'eval' / 'version', mkdir=True)
    gts, preds = load_yolo_gts_and_preds(gt_path=gt_path, pred_path=pred_path, class_names=class_names)

    # run evaluation
    for cls_name in class_names:
        preds[cls_name] = sorted(preds[cls_name], key=lambda x: -x['conf'])
    eval_results = {
        'PR_eval': PR_eval(gts, preds, class_names, iou_thr=iou_thresh),
        'confusion_matrix': confusion_matrix(gts, preds, class_names, iou_thr=iou_thresh)
    }

    # cache results
    for cls_name in class_names:
        for i in range(len(preds[cls_name])):
            preds[cls_name][i]['bbox'] = preds[cls_name][i]['bbox'].tolist()
            preds[cls_name][i]['conf'] = float(preds[cls_name][i]['conf'])

        for img_file in gts[cls_name]:
            gts[cls_name][img_file]['bbox'] = gts[cls_name][img_file]['bbox'].tolist()
            gts[cls_name][img_file]['check'] = len(gts[cls_name][img_file]['bbox']) * [False]

    with open(exp_path / 'cache_results.json', mode='w') as file:
        json.dump(obj={'preds': preds, 'gts': gts}, fp=file, indent=4)

    cache_eval_results = deepcopy(eval_results)
    for cls_name in (class_names + ['Avg.']):
        for metric_name in ['rec_array', 'prec_array', 'rec', 'prec', 'ap', 'f1']:
            if cache_eval_results['PR_eval'][cls_name][metric_name].size > 1:
                cache_eval_results['PR_eval'][cls_name][metric_name] = cache_eval_results['PR_eval'][cls_name][metric_name].tolist()
            else:
                cache_eval_results['PR_eval'][cls_name][metric_name] = float(cache_eval_results['PR_eval'][cls_name][metric_name])
    for cls_name in (class_names + ['BACKGROUND']):
        cache_eval_results['confusion_matrix'][cls_name] = cache_eval_results['confusion_matrix'][cls_name].tolist()
    with open(exp_path / 'cache_eval_results.json', mode='w') as file:
        json.dump(obj=cache_eval_results, fp=file, indent=4)

    return eval_results


## 2.1 Evaluate YOLOv8

In [ ]:
file_type = 'last'
_ = YOLO(model=os.path.join(os.getcwd(), f'runs/yolov8m/train/weights/{file_type}.pt')).predict(
    source=os.path.join(ROOT_DIR, 'test', 'images'), project=os.path.join(os.getcwd(), f'runs/yolov8m/{file_type}'),
    device=0, iou=CONF_THRESH, conf=NMS_THRESH, save=True, save_txt=True, save_conf=True
)
eval_results = yolo_evaluation(
    gt_path=os.path.join(ROOT_DIR, 'test', 'labels'),
    pred_path=os.path.join(os.getcwd(), f'runs/yolov8m/{file_type}/predict/labels'),
    class_names= CLASSES, iou_thresh=IOU_THRESH, project=os.path.join(os.getcwd(), f'runs/yolov8m/{file_type}')
)


In [ ]:
pd.DataFrame(eval_results['PR_eval']).T[['rec', 'prec', 'ap', 'f1']]


In [ ]:
# Visualize Results
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Draw PR curve metric
ax[0].plot(
    100 * eval_results['PR_eval']['Avg.']['rec_array'], 100 * eval_results['PR_eval']['Avg.']['prec_array'],
    label='YOLOv8 {} - mAP@{}: {:.3f}'.format(file_type, round(100 * IOU_THRESH), eval_results['PR_eval']['Avg.']['ap'])
)
ax[0].set_xlabel('Recall')
ax[0].set_ylabel('Precision')
ax[0].set_title('PR Curve')
ax[0].set_xlim(0, 100.1)
ax[0].set_ylim(0, 100.5)
ax[0].legend(loc='lower left')

# Draw confusion matrix
confusion_mat = eval_results['confusion_matrix']
sns.heatmap(np.array(list(confusion_mat.values())), cmap='Blues', annot=True, xticklabels=confusion_mat.keys(), yticklabels=confusion_mat.keys(), ax=ax[1])
ax[1].set_xlabel('Ground Truth')
ax[1].set_ylabel('Prediction')
ax[1].set_title(f'Confusion Matrix of YOLOv8 {file_type}')

plt.show()


In [ ]:
file_type = 'best'
_ = YOLO(model=os.path.join(os.getcwd(), f'runs/yolov8m/train/weights/{file_type}.pt')).predict(
    source=os.path.join(ROOT_DIR, 'test', 'images'), project=os.path.join(os.getcwd(), f'runs/yolov8m/{file_type}'),
    device=0, iou=CONF_THRESH, conf=NMS_THRESH, save=True, save_txt=True, save_conf=True
)
eval_results = yolo_evaluation(
    gt_path=os.path.join(ROOT_DIR, 'test', 'labels'),
    pred_path=os.path.join(os.getcwd(), f'runs/yolov8m/{file_type}/predict/labels'),
    class_names= CLASSES, iou_thresh=IOU_THRESH, project=os.path.join(os.getcwd(), f'runs/yolov8m/{file_type}')
)


In [ ]:
pd.DataFrame(eval_results['PR_eval']).T[['rec', 'prec', 'ap', 'f1']]


In [ ]:
# Visualize Results
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Draw PR curve metric
ax[0].plot(
    100 * eval_results['PR_eval']['Avg.']['rec_array'], 100 * eval_results['PR_eval']['Avg.']['prec_array'],
    label='YOLOv8 {} - mAP@{}: {:.3f}'.format(file_type, round(100 * IOU_THRESH), eval_results['PR_eval']['Avg.']['ap'])
)
ax[0].set_xlabel('Recall')
ax[0].set_ylabel('Precision')
ax[0].set_title('PR Curve')
ax[0].set_xlim(0, 100.1)
ax[0].set_ylim(0, 100.5)
ax[0].legend(loc='lower left')

# Draw confusion matrix
confusion_mat = eval_results['confusion_matrix']
sns.heatmap(np.array(list(confusion_mat.values())), cmap='Blues', annot=True, xticklabels=confusion_mat.keys(), yticklabels=confusion_mat.keys(), ax=ax[1])
ax[1].set_xlabel('Ground Truth')
ax[1].set_ylabel('Prediction')
ax[1].set_title(f'Confusion Matrix of YOLOv8 {file_type}')

plt.show()


## 2.2 Evaluate YOLOv11

In [ ]:
file_type = 'last'
_ = YOLO(model=os.path.join(os.getcwd(), f'runs/yolo11m/train/weights/{file_type}.pt')).predict(
    source=os.path.join(ROOT_DIR, 'test', 'images'), project=os.path.join(os.getcwd(), f'runs/yolo11m/{file_type}'),
    device=0, iou=CONF_THRESH, conf=NMS_THRESH, save=True, save_txt=True, save_conf=True
)
eval_results = yolo_evaluation(
    gt_path=os.path.join(ROOT_DIR, 'test', 'labels'),
    pred_path=os.path.join(os.getcwd(), f'runs/yolo11m/{file_type}/predict/labels'),
    class_names= CLASSES, iou_thresh=IOU_THRESH, project=os.path.join(os.getcwd(), f'runs/yolo11m/{file_type}')
)


In [ ]:
pd.DataFrame(eval_results['PR_eval']).T[['rec', 'prec', 'ap', 'f1']]


In [ ]:
# Visualize Results
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Draw PR curve metric
ax[0].plot(
    100 * eval_results['PR_eval']['Avg.']['rec_array'], 100 * eval_results['PR_eval']['Avg.']['prec_array'],
    label='YOLOv11 {} - mAP@{}: {:.3f}'.format(file_type, round(100 * IOU_THRESH), eval_results['PR_eval']['Avg.']['ap'])
)
ax[0].set_xlabel('Recall')
ax[0].set_ylabel('Precision')
ax[0].set_title('PR Curve')
ax[0].set_xlim(0, 100.1)
ax[0].set_ylim(0, 100.5)
ax[0].legend(loc='lower left')

# Draw confusion matrix
confusion_mat = eval_results['confusion_matrix']
sns.heatmap(np.array(list(confusion_mat.values())), cmap='Blues', annot=True, xticklabels=confusion_mat.keys(), yticklabels=confusion_mat.keys(), ax=ax[1])
ax[1].set_xlabel('Ground Truth')
ax[1].set_ylabel('Prediction')
ax[1].set_title(f'Confusion Matrix of YOLOv11 {file_type}')

plt.show()


In [ ]:
file_type = 'best'
_ = YOLO(model=os.path.join(os.getcwd(), f'runs/yolo11m/train/weights/{file_type}.pt')).predict(
    source=os.path.join(ROOT_DIR, 'test', 'images'), project=os.path.join(os.getcwd(), f'runs/yolo11m/{file_type}'),
    device=0, iou=CONF_THRESH, conf=NMS_THRESH, save=True, save_txt=True, save_conf=True
)
eval_results = yolo_evaluation(
    gt_path=os.path.join(ROOT_DIR, 'test', 'labels'),
    pred_path=os.path.join(os.getcwd(), f'runs/yolo11m/{file_type}/predict/labels'),
    class_names= CLASSES, iou_thresh=IOU_THRESH, project=os.path.join(os.getcwd(), f'runs/yolo11m/{file_type}')
)


In [ ]:
pd.DataFrame(eval_results['PR_eval']).T[['rec', 'prec', 'ap', 'f1']]


In [ ]:
# Visualize Results
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Draw PR curve metric
ax[0].plot(
    100 * eval_results['PR_eval']['Avg.']['rec_array'], 100 * eval_results['PR_eval']['Avg.']['prec_array'],
    label='YOLOv11 {} - mAP@{}: {:.3f}'.format(file_type, round(100 * IOU_THRESH), eval_results['PR_eval']['Avg.']['ap'])
)
ax[0].set_xlabel('Recall')
ax[0].set_ylabel('Precision')
ax[0].set_title('PR Curve')
ax[0].set_xlim(0, 100.1)
ax[0].set_ylim(0, 100.5)
ax[0].legend(loc='lower left')

# Draw confusion matrix
confusion_mat = eval_results['confusion_matrix']
sns.heatmap(np.array(list(confusion_mat.values())), cmap='Blues', annot=True, xticklabels=confusion_mat.keys(), yticklabels=confusion_mat.keys(), ax=ax[1])
ax[1].set_xlabel('Ground Truth')
ax[1].set_ylabel('Prediction')
ax[1].set_title(f'Confusion Matrix of YOLOv11 {file_type}')

plt.show()
